# Numerical Evaluation: Optimal and Greedy Sensor Allocation

**Run this notebook using Kernel > Restart Kernel and Run All Cells.** It is the single interactive entry point for the complete experiment.

This notebook runs the paper's **Exact Dynamic Programming** and **Greedy Gain-Based Approximation** methods on the same generated inputs. The Python files are supporting modules imported by this notebook and do not need to be run separately.

## Benchmark datasets

The evaluation uses two separate datasets so that one- and two-dimensional results remain distinguishable.

| Dataset | Region | Cases |
|---|---|---:|
| `benchmark_1d` | $[0,1]$ | 5,760 |
| `benchmark_2d` | $[0,1]^2$ | 5,760 |

Each dataset contains three field families, $K=3,\ldots,10$ candidate fields, $M=3,\ldots,10$ candidate sensing locations, and 30 independent realizations per family–$K$–$M$ cell. The phase scale is fixed at $c=1$.

## Field and location generation

| Family | Main parameter ranges |
|---|---|
| Gaussian | amplitude $[0.5,1.5]$, centre in the sensing region, width $[0.1,0.3]$ |
| Sinusoidal | amplitude $[0.5,1.5]$, frequency $\{1,2,3\}$, phase $[-\pi,\pi]$, and 2D orientation $[0,\pi)$ |
| Quadratic | background $[-0.5,0.5]$, curvature $[-1,1]$, centre in the sensing region |

For every realization, the field parameters and candidate locations are sampled again. Locations are independent uniform samples from the relevant region. All $K$ fields in one case use the same $M$ locations. The exact formulas and every realized value are recorded in the dataset manifest.

## Load the evaluation pipeline

The next cell locates the project directory whether the notebook is launched from the repository root or from this folder.

In [42]:
from pathlib import Path
import sys

candidates = [Path.cwd() / "min-sensor-zero-error", Path.cwd()]
project_dir = next(
    candidate.resolve()
    for candidate in candidates
    if (candidate / "evaluate_benchmarks.py").is_file()
)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

# Discard local modules cached by an earlier notebook run. This ensures
# that the current pipeline and figure definitions are always imported.
local_modules = (
    "optimal", "greedy", "phase_data", "field_models",
    "generate_inputs", "generate_benchmarks", "eval",
    "paper_analysis", "evaluate_benchmarks",
)
for module_name in local_modules:
    sys.modules.pop(module_name, None)

from evaluate_benchmarks import run_benchmark_evaluation

project_dir

PosixPath('/Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error')

## Generate and evaluate both benchmarks

On the first run, this cell generates any missing benchmark files. It then runs both algorithms on all 11,520 cases, calculates the statistical summaries, and exports the figures. Later runs reuse the generated inputs. Runtime repetitions are set to one because the allocations are deterministic; increase `repeats` only when collecting final timing measurements.

In [43]:
analysis = run_benchmark_evaluation(
    repeats=1,
    bootstrap_iterations=2_000,
)

expected_figures = {"heatmaps", "families", "guarantee"}
if set(analysis["figures"]) != expected_figures:
    raise RuntimeError("The current three-figure analysis was not loaded.")

Evaluating benchmark_1d ...
  5760 cases -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/output/benchmark_1d
Evaluating benchmark_2d ...
  5760 cases -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/output/benchmark_2d
Benchmark tables and figures -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/output/benchmark-analysis


## Numerical summary

The first table reports the combined result. The second table keeps the 1D and 2D benchmarks separate, which should be the main presentation in the paper.

In [44]:
from IPython.display import Markdown, display

def summary_table(rows):
    header = (
        "| Group | Cases | Mean ratio | 95% CI | Exact-match rate | "
        "95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = []
    for row in rows:
        body.append(
            f"| {row['group_value']} | {row['case_count']} | "
            f"{row['mean_greedy_ratio']:.3f} | "
            f"[{row['mean_greedy_ratio_ci95_low']:.3f}, "
            f"{row['mean_greedy_ratio_ci95_high']:.3f}] | "
            f"{100 * row['exact_match_rate']:.1f}% | "
            f"{row['p95_greedy_ratio']:.3f} | "
            f"{row['max_greedy_ratio']:.3f} |"
        )
    return header + "\n" + "\n".join(body)

display(Markdown(summary_table([analysis['overall_summary']])))
display(Markdown(summary_table(analysis['dimension_summaries'])))

| Group | Cases | Mean ratio | 95% CI | Exact-match rate | 95th percentile | Maximum |
|---|---:|---:|---:|---:|---:|---:|
| combined | 11520 | 1.152 | [1.149, 1.155] | 30.1% | 1.487 | 2.280 |

| Group | Cases | Mean ratio | 95% CI | Exact-match rate | 95th percentile | Maximum |
|---|---:|---:|---:|---:|---:|---:|
| 1D | 5760 | 1.161 | [1.157, 1.165] | 28.6% | 1.500 | 2.280 |
| 2D | 5760 | 1.144 | [1.141, 1.148] | 31.6% | 1.478 | 2.164 |

## Mean sensor excess across $K$ and $M$

The two panels show the 1D and 2D heatmaps using the same colour scale.

In [45]:
display(analysis["figures"]["heatmaps"]);

<Figure size 1150x480 with 3 Axes>

## Sensor excess by field family

The panels show $K=3,\ldots,10$ on the horizontal axis, with separate Gaussian, sinusoidal, and quadratic bars at each $K$. A bar is the mean percentage of additional sensors used by greedy over 240 cases: eight values of $M$ and 30 realizations. Error bars are 95% bootstrap intervals for the mean, stratified by $M$.

In [46]:
display(analysis["figures"]["families"]);

<Figure size 1150x440 with 2 Axes>

## Empirical performance and approximation guarantee

Let $B(K)=1+\ln\binom{K}{2}$ be the approximation factor and $\rho=N_{\mathrm{greedy}}/N_{\mathrm{optimal}}$. The plotted measure is $100(\rho-1)/(B(K)-1)$, the percentage of the theorem's permitted additional-sensor excess actually used. For each family and $K$, the colored line is the maximum observed value over 240 cases. The dashed line at 100% is the worst-case guarantee limit. These sample maxima describe the generated benchmarks and are not estimates of the worst possible instance outside them.

In [47]:
display(analysis["figures"]["guarantee"]);

<Figure size 1150x420 with 2 Axes>

## Paper wording generated from the current results

In [48]:
findings_path = Path(analysis["output_directory"]) / "paper_findings.md"
display(Markdown(findings_path.read_text(encoding="utf-8")))

# Empirical findings template

Across the two pre-specified synthetic benchmarks (n = 11520), the greedy allocation matched the exact optimum in 30.1% of cases.
Its mean greedy-to-optimal sensor ratio was 1.152 (95% stratified-bootstrap CI 1.149--1.155); the 95th percentile was 1.487.

For the 1D benchmark (n = 5760), the mean ratio was 1.161 and the exact-match rate was 28.6%.

For the 2D benchmark (n = 5760), the mean ratio was 1.144 and the exact-match rate was 31.6%.

These results describe the stated normalized synthetic benchmark distribution. They do not strengthen the worst-case guarantee or establish performance for every physical sensing application.
